# OOV rate per split

How many tokens of the dataset have no word2vec vector, using the same tokenization and lookup as training (`art.preprocess`):

- Tokens come from `nltk.word_tokenize`; punctuation-only tokens are dropped.
- A token is **found** if `lookup_token` finds a vector: exact match, then digits replaced by `#`, then lowercased.
- **exact** is the OOV rate without these fallbacks, to show what they gain.
- The train split is the filtered one from `get_dataset` (empty hypotheses removed), i.e. what the models are trained on.

In [ ]:
import re

import nltk
import polars as pl

from art.data import get_dataset
from art.embedding import COLS
from art.models import load_w2v
from art.preprocess import lookup_token

pl.Config.set_tbl_rows(30)
pl.Config.set_tbl_cols(-1)
pl.Config.set_tbl_width_chars(250)

In [ ]:
wv = load_w2v()
ds = get_dataset()

In [ ]:
rows = []
text_id = 0
for split, d in ds.items():
    for col in COLS:
        for text in d[col]:
            text_id += 1
            # same token filter as art.preprocess.to_tokens
            for t in nltk.word_tokenize(text):
                if re.fullmatch(r"\W+", t):
                    continue
                rows.append(
                    {
                        "split": split,
                        "column": col,
                        "text_id": text_id,
                        "token": t,
                        "exact": t in wv,
                        "found": lookup_token(t, wv) is not None,
                    }
                )

tokens = pl.DataFrame(rows)
tokens.shape

In [ ]:
def oov_stats(df: pl.DataFrame, by: list[str]) -> pl.DataFrame:
    texts = (
        df.group_by(by + ["text_id"])
        .agg(has_oov=(~pl.col("found")).any())
        .group_by(by)
        .agg(texts_with_oov_pct=pl.col("has_oov").mean() * 100)
    )
    return (
        df.group_by(by, maintain_order=True)
        .agg(
            tokens=pl.len(),
            oov_tokens=(~pl.col("found")).sum(),
            oov_pct=(~pl.col("found")).mean() * 100,
            oov_pct_exact=(~pl.col("exact")).mean() * 100,
            types=pl.col("token").n_unique(),
            oov_types=pl.col("token").filter(~pl.col("found")).n_unique(),
        )
        .with_columns(oov_types_pct=pl.col("oov_types") / pl.col("types") * 100)
        .join(texts, on=by, maintain_order="left")
        .with_columns(pl.selectors.float().round(2))
    )

## Per split

- `oov_pct`: share of tokens without a vector (these become `<unk>`)
- `oov_pct_exact`: same, without the digit/lowercase fallbacks
- `oov_types_pct`: share of distinct tokens without a vector
- `texts_with_oov_pct`: share of texts (single observation/hypothesis) with at least one OOV token

In [ ]:
per_split = pl.concat(
    [
        oov_stats(tokens, ["split"]),
        oov_stats(tokens.with_columns(split=pl.lit("all")), ["split"]),
    ]
)
per_split

## Per split and column

In [ ]:
oov_stats(tokens, ["split", "column"])

## Most frequent OOV tokens

In [ ]:
(
    tokens.filter(~pl.col("found"))
    .group_by("token")
    .agg(count=pl.len(), splits=pl.col("split").unique().sort())
    .sort("count", descending=True)
    .head(25)
)

## OOV without function words missing from word2vec

The GoogleNews word2vec model deliberately has no vectors for a few very frequent function words (`a`, `to`, `and`, `of`), and the clitic `'s` that `nltk` splits off (`John's` → `John`, `'s`) is missing too. These make up almost all OOV tokens, so the rate of **content words** without a vector is much lower.

In [ ]:
missing_function_words = ["a", "to", "and", "of", "'s"]
assert not any(lookup_token(w, wv) for w in missing_function_words)

is_function_word = pl.col("token").str.to_lowercase().is_in(missing_function_words)
oov = tokens.filter(~pl.col("found"))
print(
    f"function words make up {oov.select(is_function_word.mean()).item() * 100:.1f}% of all OOV tokens"
)

content = tokens.filter(~is_function_word)
pl.concat(
    [
        oov_stats(content, ["split"]),
        oov_stats(content.with_columns(split=pl.lit("all")), ["split"]),
    ]
)